# 03 - Feedforward Neural Network (Simple Upload Version)

This notebook trains the neural network model for both tasks using **TensorFlow / Keras**.

It is written to be simple and Colab-friendly:

- manually upload the dataset files you need
- minimal path logic
- save outputs automatically
- automatically download a zip of the saved outputs at the end

## Files you will upload
- `taskA_dataset.csv`
- `taskB_dataset.csv`

## Files this notebook saves
- `taskA_ffnn_results.csv`
- `taskB_ffnn_results.csv`
- `taskA_preds_ffnn.csv`
- `taskB_preds_ffnn.csv`
- training-history plots for Task A and Task B
- one zip file containing all FFNN outputs


In [ ]:

# Basic imports
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

pd.set_option('display.max_columns', 200)
np.random.seed(42)
tf.random.set_seed(42)


## 1) File setup

Upload `taskA_dataset.csv` and `taskB_dataset.csv` when prompted.


In [ ]:

from google.colab import files

OUTPUT_DIR = '/content/ds4400_valorant_ffnn_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Please upload taskA_dataset.csv and taskB_dataset.csv")
uploaded = files.upload()

taskA = pd.read_csv('taskA_dataset.csv', parse_dates=['Date'])
taskB = pd.read_csv('taskB_dataset.csv', parse_dates=['Date'])

print("Task A shape:", taskA.shape)
print("Task B shape:", taskB.shape)
taskA.head(2)


## 2) Helper functions

In [ ]:

RANDOM_STATE = 42
FAST_MODE = False        # set True only if runtime is becoming a problem
MAX_TRAIN_ROWS = 60000   # used only when FAST_MODE = True

def split_train_test(df, target_col='Team1 Win'):
    train_df = df[df['Split'] == 'train'].copy()
    test_df = df[df['Split'] == 'test'].copy()

    X_train = train_df.drop(columns=['Date', 'Split', target_col], errors='ignore')
    y_train = train_df[target_col].copy().astype(int)

    X_test = test_df.drop(columns=['Date', 'Split', target_col], errors='ignore')
    y_test = test_df[target_col].copy().astype(int)

    return X_train, X_test, y_train, y_test

def maybe_sample_train(X, y, max_rows=60000, random_state=42):
    if len(X) <= max_rows:
        return X, y
    idx = y.sample(n=max_rows, random_state=random_state).index
    return X.loc[idx].copy(), y.loc[idx].copy()

def make_preprocessor(X):
    cat_cols = [c for c in X.columns if c == 'Map']
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), cat_cols)
        ],
        remainder='drop'
    )

    return preprocessor

def build_ffnn(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dropout(0.25),
        Dense(64, activation='relu'),
        Dropout(0.20),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def evaluate_predictions(y_true, preds, probas, task_name):
    return {
        'Task': task_name,
        'Model': 'FFNN',
        'Accuracy': accuracy_score(y_true, preds),
        'Precision': precision_score(y_true, preds, zero_division=0),
        'Recall': recall_score(y_true, preds, zero_division=0),
        'F1': f1_score(y_true, preds, zero_division=0),
        'ROC_AUC': roc_auc_score(y_true, probas),
        'Brier': brier_score_loss(y_true, probas)
    }

def run_ffnn_for_task(df, task_name):
    X_train, X_test, y_train, y_test = split_train_test(df)

    if FAST_MODE:
        X_train, y_train = maybe_sample_train(X_train, y_train, MAX_TRAIN_ROWS, RANDOM_STATE)

    preprocessor = make_preprocessor(X_train)
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)

    if hasattr(X_train_proc, "toarray"):
        X_train_proc = X_train_proc.toarray()
    if hasattr(X_test_proc, "toarray"):
        X_test_proc = X_test_proc.toarray()

    model = build_ffnn(X_train_proc.shape[1])
    model.summary()

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_proc, y_train.values,
        validation_split=0.2,
        epochs=20,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )

    probas = model.predict(X_test_proc, verbose=0).ravel()
    preds = (probas >= 0.5).astype(int)

    results = evaluate_predictions(y_test.values, preds, probas, task_name)
    pred_df = pd.DataFrame({
        'y_true': y_test.values,
        'y_pred': preds,
        'y_prob': probas
    })

    return results, pred_df, history


## 3) Run Task A FFNN

In [ ]:

taskA_results, taskA_pred_df, history_A = run_ffnn_for_task(taskA, 'Task A')
taskA_results_df = pd.DataFrame([taskA_results])
taskA_results_df


## 4) Run Task B FFNN

In [ ]:

taskB_results, taskB_pred_df, history_B = run_ffnn_for_task(taskB, 'Task B')
taskB_results_df = pd.DataFrame([taskB_results])
taskB_results_df


## 5) Save outputs and plots

In [ ]:

# Save CSV outputs
taskA_results_path = os.path.join(OUTPUT_DIR, 'taskA_ffnn_results.csv')
taskB_results_path = os.path.join(OUTPUT_DIR, 'taskB_ffnn_results.csv')
taskA_preds_path = os.path.join(OUTPUT_DIR, 'taskA_preds_ffnn.csv')
taskB_preds_path = os.path.join(OUTPUT_DIR, 'taskB_preds_ffnn.csv')

taskA_results_df.to_csv(taskA_results_path, index=False)
taskB_results_df.to_csv(taskB_results_path, index=False)
taskA_pred_df.to_csv(taskA_preds_path, index=False)
taskB_pred_df.to_csv(taskB_preds_path, index=False)

# Save history plots
def save_history_plot(history, task_name, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{task_name} FFNN Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Binary Crossentropy')
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    return path

taskA_plot_path = save_history_plot(history_A, 'Task A', 'taskA_ffnn_loss.png')
taskB_plot_path = save_history_plot(history_B, 'Task B', 'taskB_ffnn_loss.png')

print("Saved:")
for p in [taskA_results_path, taskB_results_path, taskA_preds_path, taskB_preds_path, taskA_plot_path, taskB_plot_path]:
    print(p)


## 6) Auto-download one zip with all FFNN outputs

In [ ]:

import zipfile
from google.colab import files

zip_path = '/content/ffnn_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        full_path = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(full_path):
            zf.write(full_path, arcname=fname)

print("Downloading:", zip_path)
files.download(zip_path)
